# 树、二叉树与森林的相互转化

普通树中的一个节点可以有任意多个孩子，而二叉树中的一个节点最多只有左、右两个孩子。为了用二叉树表示普通树或森林，需要重新解释二叉树中两个指针的含义。最常用的方法是**左孩子右兄弟表示法**，也叫 **Left-Child Right-Sibling, LCRS** 表示法。

它的核心原则只有一句话：

- `left` 指向该节点在原树中的**第一个孩子**；
- `right` 指向该节点在原树中的**下一个兄弟**。

因此，转换后的二叉树并不是普通意义上“左子树小、右子树大”或“左右孩子都是真正子节点”的二叉树。这里的 `right` 指针更多表示同一层中的兄弟关系，而不是原树中的父子关系。

## 1. 树与二叉树之间的转化

设普通树中某个节点 `A` 有若干个孩子，按从左到右的顺序分别为：

```text
A 的孩子：B, C, D, E
```

按照“左孩子右兄弟”原则转化为二叉树时：

1. `A.left = B`，也就是把第一个孩子 `B` 放到 `A` 的左指针上。
2. `B.right = C`，因为 `C` 是 `B` 的下一个兄弟。
3. `C.right = D`，因为 `D` 是 `C` 的下一个兄弟。
4. `D.right = E`，因为 `E` 是 `D` 的下一个兄弟。
5. 对 `B, C, D, E` 各自的子树继续递归执行同样的规则。

转换后的局部结构可以理解为：

```text
普通树：

        A
     /  |  |  \
    B   C  D   E

二叉树表示：

        A
       /
      B
       \
        C
         \
          D
           \
            E
```

需要注意：在这个二叉树中，`B, C, D, E` 仍然都是普通树里 `A` 的孩子，只是除了第一个孩子 `B` 通过 `A.left` 连接外，其余孩子都挂在右兄弟链上。

### 普通树转二叉树的步骤

对普通树中每个节点执行以下操作：

1. 如果节点没有孩子，则它在二叉树中的 `left` 指针为空。
2. 如果节点有孩子，则第一个孩子成为它的 `left` 节点。
3. 从第二个孩子开始，依次作为前一个孩子的 `right` 节点，形成一条右兄弟链。
4. 对每个孩子节点递归处理它自己的孩子列表。

### 二叉树还原为普通树的步骤

如果一棵二叉树本来就是由普通树按 LCRS 方法转换得到的，那么可以反向还原：

1. 对于当前二叉树节点 `A`，它对应普通树中的同名节点 `A`。
2. 沿着 `A.left` 找到它的第一个孩子。
3. 从这个第一个孩子开始，不断沿 `right` 指针向右走，得到 `A` 的全部孩子。
4. 对这些孩子递归还原它们自己的孩子。
5. 如果正在还原的是一棵普通树，那么根节点的 `right` 指针应该为空；如果根节点的 `right` 不为空，说明它更像是在表示一个森林。

In [1]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any


@dataclass
class TreeNode:
    """普通树节点：一个节点可以有任意多个孩子。"""

    value: Any
    children: list["TreeNode"] = field(default_factory=list)

    def add_child(self, child: "TreeNode") -> "TreeNode":
        self.children.append(child)
        return child


@dataclass
class BinaryNode:
    """二叉树节点：left 表示第一个孩子，right 表示下一个兄弟。"""

    value: Any
    left: "BinaryNode | None" = None
    right: "BinaryNode | None" = None


def _encode_tree_node(root: TreeNode | None) -> BinaryNode | None:
    """把一棵普通树编码为 LCRS 二叉树，不处理根节点的兄弟。"""

    if root is None:
        return None

    binary_root = BinaryNode(root.value)

    if root.children:
        binary_root.left = _encode_tree_node(root.children[0])
        current = binary_root.left

        for child in root.children[1:]:
            current.right = _encode_tree_node(child)
            current = current.right

    return binary_root


def tree_to_binary(root: TreeNode | None) -> BinaryNode | None:
    """普通树 -> 二叉树。"""

    return _encode_tree_node(root)


def _decode_tree_node(root: BinaryNode | None) -> TreeNode | None:
    """把一个 LCRS 二叉树节点还原为普通树节点，不处理该节点自己的兄弟。"""

    if root is None:
        return None

    tree_root = TreeNode(root.value)
    child = root.left

    while child is not None:
        tree_root.children.append(_decode_tree_node(child))
        child = child.right

    return tree_root


def binary_to_tree(root: BinaryNode | None) -> TreeNode | None:
    """二叉树 -> 普通树。要求根节点没有右兄弟。"""

    if root is not None and root.right is not None:
        raise ValueError("根节点存在 right 指针，当前二叉树表示的是森林，请使用 binary_to_forest")

    return _decode_tree_node(root)


def print_general_tree(root: TreeNode | None, indent: str = "") -> None:
    """按缩进打印普通树。"""

    if root is None:
        print("<empty>")
        return

    print(f"{indent}{root.value}")
    for child in root.children:
        print_general_tree(child, indent + "  ")


def print_lcrs_binary_tree(root: BinaryNode | None, indent: str = "", label: str = "root") -> None:
    """打印 LCRS 二叉树，并标明 left/right 的含义。"""

    if root is None:
        print(f"{indent}{label}: <empty>")
        return

    print(f"{indent}{label}: {root.value}")
    if root.left is not None:
        print_lcrs_binary_tree(root.left, indent + "  ", "left(first child)")
    if root.right is not None:
        print_lcrs_binary_tree(root.right, indent + "  ", "right(next sibling)")


In [2]:
# 示例：普通树 -> 二叉树 -> 普通树
#
# 普通树结构：
#        A
#      / | \
#     B  C  D
#    / \    |
#   E   F   G

root = TreeNode("A")
node_b = root.add_child(TreeNode("B"))
root.add_child(TreeNode("C"))
node_d = root.add_child(TreeNode("D"))
node_b.add_child(TreeNode("E"))
node_b.add_child(TreeNode("F"))
node_d.add_child(TreeNode("G"))

binary_root = tree_to_binary(root)
restored_root = binary_to_tree(binary_root)

print("原普通树：")
print_general_tree(root)

print("\n转换后的 LCRS 二叉树：")
print_lcrs_binary_tree(binary_root)

print("\n还原后的普通树：")
print_general_tree(restored_root)


原普通树：
A
  B
    E
    F
  C
  D
    G

转换后的 LCRS 二叉树：
root: A
  left(first child): B
    left(first child): E
      right(next sibling): F
    right(next sibling): C
      right(next sibling): D
        left(first child): G

还原后的普通树：
A
  B
    E
    F
  C
  D
    G


## 2. 森林与二叉树之间的转化

森林是若干棵互不相交的树的集合。设一个森林由三棵树组成，它们的根节点分别是：

```text
T1 的根：A
T2 的根：H
T3 的根：K
```

森林转二叉树时，仍然使用“左孩子右兄弟”原则，只是需要把不同树的根节点也看成同一层的兄弟：

1. 先把每棵普通树分别按 LCRS 方法转换成二叉树。
2. 第一棵树的根 `A` 作为整棵二叉树的根。
3. 第二棵树的根 `H` 作为 `A.right`，表示 `H` 是 `A` 的下一个兄弟。
4. 第三棵树的根 `K` 作为 `H.right`，继续形成右兄弟链。
5. 每棵树根节点自己的孩子仍然通过 `left` 指针进入。

也就是说，森林中的多棵树，在二叉树中会表现为一条从根节点开始的 `right` 链：

```text
森林：

    A          H          K
   / \        /          / \
  B   C      I          L   M

二叉树表示：

    A
   / \
  B   H
   \  / \
    C I  K
        /
       L
        \
         M
```

在这张二叉树里：

- `A.right = H` 表示森林中第二棵树的根；
- `H.right = K` 表示森林中第三棵树的根；
- `A.left, H.left, K.left` 分别进入每棵树自己的第一个孩子；
- 根节点之间的 `right` 链表示森林中不同树根之间的兄弟关系。

### 森林转二叉树的步骤

1. 如果森林为空，则转换结果为空二叉树。
2. 把森林中第一棵树的根作为二叉树的根。
3. 对每棵树内部按“第一个孩子放 left，其余孩子放 right 兄弟链”的方式转换。
4. 把森林中第二棵、第三棵、后续每棵树的根依次连接到前一棵树根的 `right` 指针上。

### 二叉树还原为森林的步骤

1. 从二叉树根节点开始，沿着 `right` 指针依次访问，得到森林中每棵树的根。
2. 对每个根节点，沿着它的 `left` 指针进入第一个孩子。
3. 对每个孩子，再沿着 `right` 指针找到它的所有兄弟，即得到同一个父节点的全部孩子。
4. 递归还原所有子树。

普通树与森林的区别主要体现在根节点的 `right` 指针上：普通树转换成的二叉树根节点通常没有 `right`，而森林转换成的二叉树根节点可能有 `right`，用来连接森林中的下一棵树。

In [3]:
def forest_to_binary(forest: list[TreeNode]) -> BinaryNode | None:
    """森林 -> 二叉树。森林中各棵树的根节点通过 right 指针连接。"""

    if not forest:
        return None

    binary_root = _encode_tree_node(forest[0])
    current = binary_root

    for tree_root in forest[1:]:
        current.right = _encode_tree_node(tree_root)
        current = current.right

    return binary_root


def binary_to_forest(root: BinaryNode | None) -> list[TreeNode]:
    """二叉树 -> 森林。二叉树根节点及其 right 链对应森林的各棵树根。"""

    forest = []
    current = root

    while current is not None:
        forest.append(_decode_tree_node(current))
        current = current.right

    return forest


def print_forest(forest: list[TreeNode]) -> None:
    """按顺序打印森林中的每棵树。"""

    if not forest:
        print("<empty forest>")
        return

    for index, tree_root in enumerate(forest, start=1):
        print(f"Tree {index}:")
        print_general_tree(tree_root, "  ")


In [4]:
# 示例：森林 -> 二叉树 -> 森林
#
# 森林结构：
#
#     A        H        K
#    / \      /        / \
#   B   C    I        L   M

tree1 = TreeNode("A", [TreeNode("B"), TreeNode("C")])
tree2 = TreeNode("H", [TreeNode("I")])
tree3 = TreeNode("K", [TreeNode("L"), TreeNode("M")])

forest = [tree1, tree2, tree3]
binary_forest_root = forest_to_binary(forest)
restored_forest = binary_to_forest(binary_forest_root)

print("原森林：")
print_forest(forest)

print("\n转换后的 LCRS 二叉树：")
print_lcrs_binary_tree(binary_forest_root)

print("\n还原后的森林：")
print_forest(restored_forest)


原森林：
Tree 1:
  A
    B
    C
Tree 2:
  H
    I
Tree 3:
  K
    L
    M

转换后的 LCRS 二叉树：
root: A
  left(first child): B
    right(next sibling): C
  right(next sibling): H
    left(first child): I
    right(next sibling): K
      left(first child): L
        right(next sibling): M

还原后的森林：
Tree 1:
  A
    B
    C
Tree 2:
  H
    I
Tree 3:
  K
    L
    M
